# Geração eólica cortada no Nordeste — média mensal de 2026

Segundo o [dicionário oficial do ONS](https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/restricao_coff_eolica_tm/DicionarioDados_RestricaoContrainedoff_UsiEolicas.json), a coluna correta é `val_geracaonaorealizadaapurada` (GNRa): geração frustrada, em MWmed, calculada pela diferença positiva entre a geração de referência e a verificada nos períodos com limitação.

`val_geracaolimitada` **não** representa o volume cortado; ela é o limite de geração estabelecido pelo ONS. O arquivo consolidado local ainda usa o esquema anterior do conjunto e não contém a GNRa pronta, então ela é reconstruída pela definição do dicionário: `max(val_geracaoreferencia - val_geracao, 0)` nos intervalos em que houve limitação. Para obter o valor do subsistema, a GNRa das usinas é somada em cada intervalo de 30 minutos e, depois, é calculada a média dos intervalos de cada mês. A energia mensal em MWh também é apresentada como informação complementar (`MWmed × 0,5 h`).

In [11]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq

ANO = 2026
SUBSISTEMA = "NE"
COLUNA_CORTE = "val_geracaonaorealizadaapurada"
NOME_ARQUIVO = "constrained_off_eolica_tm.parquet"

# Funciona tanto ao executar a partir de notebooks/ quanto da raiz do projeto.
CANDIDATOS = [
    Path("../data/coff") / NOME_ARQUIVO,
    Path("data/coff") / NOME_ARQUIVO,
    Path("../data/processed/coff") / NOME_ARQUIVO,
    Path("data/processed/coff") / NOME_ARQUIVO,
]
CAMINHO_DADOS = next((caminho for caminho in CANDIDATOS if caminho.exists()), None)

if CAMINHO_DADOS is None:
    raise FileNotFoundError(
        f"{NOME_ARQUIVO} não foi encontrado em data/coff nem em data/processed/coff."
    )

In [12]:
colunas_disponiveis = set(pq.read_schema(CAMINHO_DADOS).names)
colunas_base = ["id_subsistema", "id_ons", "din_instante"]
colunas_calculo = (
    [COLUNA_CORTE]
    if COLUNA_CORTE in colunas_disponiveis
    else ["val_geracao", "val_geracaolimitada", "val_geracaoreferencia"]
)

coff_2026 = pd.read_parquet(
    CAMINHO_DADOS,
    columns=colunas_base + colunas_calculo,
    filters=[("ano", "=", ANO), ("id_subsistema", "=", SUBSISTEMA)],
)
coff_2026["din_instante"] = pd.to_datetime(coff_2026["din_instante"])

print(
    f"Arquivo local: {CAMINHO_DADOS.resolve()}\n"
    f"dados até {coff_2026['din_instante'].max():%d/%m/%Y %H:%M}."
)

Arquivo local: /home/alves/projetos/egide/data/processed/coff/constrained_off_eolica_tm.parquet
dados até 31/08/2026 23:30.


In [13]:
if COLUNA_CORTE not in coff_2026.columns:
    # O consolidado local usa o esquema anterior. Reproduz a definição da GNRa
    # do ONS somente nos patamares em que há limite de geração informado.
    houve_limitacao = coff_2026["val_geracaolimitada"].notna()
    diferenca_positiva = (
        coff_2026["val_geracaoreferencia"] - coff_2026["val_geracao"]
    ).clip(lower=0)
    coff_2026[COLUNA_CORTE] = diferenca_positiva.where(houve_limitacao, 0.0)

coff_ne = coff_2026[["id_ons", "din_instante", COLUNA_CORTE]].copy()
coff_ne[COLUNA_CORTE] = pd.to_numeric(coff_ne[COLUNA_CORTE], errors="coerce").fillna(0.0)

if coff_ne.duplicated(["id_ons", "din_instante"]).any():
    raise ValueError("Há registros duplicados de usina e instante nos dados do ONS.")

# Primeiro soma as usinas para formar o total do Nordeste em cada meia hora.
corte_30min_ne = (
    coff_ne.groupby("din_instante", as_index=False)[COLUNA_CORTE]
    .sum()
    .rename(columns={COLUNA_CORTE: "geracao_cortada_mwmed"})
)
corte_30min_ne["mes"] = (
    corte_30min_ne["din_instante"].dt.to_period("M").dt.to_timestamp()
)

In [14]:
mensal_2026 = (
    corte_30min_ne.groupby("mes", as_index=False)
    .agg(
        geracao_cortada_media_mwmed=("geracao_cortada_mwmed", "mean"),
        energia_cortada_mwh=("geracao_cortada_mwmed", lambda s: s.sum() * 0.5),
        intervalos_30min=("din_instante", "nunique"),
        primeiro_instante=("din_instante", "min"),
        ultimo_instante=("din_instante", "max"),
    )
)
mensal_2026["intervalos_esperados"] = mensal_2026["mes"].dt.days_in_month * 48
mensal_2026["cobertura_pct"] = (
    100 * mensal_2026["intervalos_30min"] / mensal_2026["intervalos_esperados"]
)
mensal_2026["status"] = mensal_2026["cobertura_pct"].ge(100).map(
    {True: "completo", False: "parcial"}
)

resultado = mensal_2026.assign(mes=mensal_2026["mes"].dt.strftime("%Y-%m"))[
    [
        "mes",
        "geracao_cortada_media_mwmed",
        "energia_cortada_mwh",
        "cobertura_pct",
        "status",
    ]
]

resultado.style.format(
    {
        "geracao_cortada_media_mwmed": "{:,.2f}",
        "energia_cortada_mwh": "{:,.2f}",
        "cobertura_pct": "{:.1f}%",
    }
)

,mes,geracao_cortada_media_mwmed,energia_cortada_mwh,cobertura_pct,status
0,2026-01,"3,252.70","2,420,009.41",100.0%,completo
1,2026-02,726.01,"487,880.82",100.0%,completo
2,2026-03,"1,108.35","824,611.42",100.0%,completo
3,2026-04,"1,904.52","1,371,256.29",100.0%,completo
4,2026-05,"2,713.29","2,018,686.57",100.0%,completo
5,2026-06,"2,388.52","1,719,734.95",100.0%,completo
6,2026-07,"3,959.36","2,945,764.60",100.0%,completo
7,2026-08,"4,815.40","3,582,657.26",100.0%,completo


In [15]:
fig = px.bar(
    mensal_2026,
    x="mes",
    y="geracao_cortada_media_mwmed",
    color="status",
    text_auto=".1f",
    labels={
        "mes": "Mês",
        "geracao_cortada_media_mwmed": "Geração cortada média (MWmed)",
        "status": "Cobertura do mês",
    },
    title="Geração eólica cortada média mensal — Nordeste (2026)",
)
fig.update_xaxes(dtick="M1", tickformat="%b")
fig.update_layout(hovermode="x unified")
fig.show()